# CNN - CIFAR10 - Basic model

# Import Libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

import torchvision
import torchvision.transforms as transforms
import os
import sys
sys.path.insert(0, '..')
from utils import *


device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f' The Device is set to : {device}')

 The Device is set to : cuda


# Import dataset

- augmented
- normalized
- padded
- shuffled
- CIFAR10

In [2]:
transform_train = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.Resize(192),  
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.Resize(192),  
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

BATCH_SIZE = 16

trainset = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=transform_train)
trainloader = torch.utils.data.DataLoader(
    trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

testset = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=transform_test)
testloader = torch.utils.data.DataLoader(
    testset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

classes = ('plane', 'car', 'bird', 'cat', 'deer',
           'dog', 'frog', 'horse', 'ship', 'truck')

Files already downloaded and verified
Files already downloaded and verified


# CNN Model

In [3]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        self.ad_pool = nn.AdaptiveAvgPool2d(output_size=(6,6))
        
        self.fc1 = nn.Linear(in_features=128 * 6 * 6,out_features= 512, bias= True) 
        self.fc2 = nn.Linear(in_features= 512, out_features=256)
        self.fc3 = nn.Linear(256, 10) 
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = self.ad_pool(x)

        x = x.view(-1, 128 * 6 * 6)

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

# Model : CNN

In [4]:
model = CNN().to(device)

num_params: int = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("number of parameters:" , num_params)
print(model)

number of parameters: 2586954
CNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (ad_pool): AdaptiveAvgPool2d(output_size=(6, 6))
  (fc1): Linear(in_features=4608, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=256, bias=True)
  (fc3): Linear(in_features=256, out_features=10, bias=True)
)


# Model Train and Evaluation

In [5]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr= 0.1,
                        momentum=0.9, weight_decay=5e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=200)

In [6]:
def topk(output, target, k):
    correct = 0.0
    batch_size = output.size(0)
    _, topk_indices = output.topk(k, dim=1, largest=True, sorted=True)
    target = target.view(-1, 1).expand_as(topk_indices)
    
    correct += (topk_indices == target).sum().item()

    return (correct / batch_size) * 100.0
        


In [7]:

def train(epoch):
    file_path = '../results/cifar_10/basic_model/CNN_train.txt'
    
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    
    print('\nEpoch: %d' % epoch)
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    
    with open(file_path, 'a') as f:
        f.write(f'\nEpoch: {epoch}\n')
        
        for batch_idx, (inputs, targets) in enumerate(trainloader):
            inputs, targets = inputs.to(device), targets.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()

        train_summary = f'Train Summary after Epoch: {epoch}, Loss: {train_loss / len(trainloader):.3f}, Accuracy: {100. * correct / total:.3f}% ({correct}/{total})\n'
        f.write(train_summary)

        print(train_summary)
        model_save_path = f'../results/cifar_10/basic_model/epoch_{epoch}_Cnn_train.pth'
        os.makedirs(os.path.dirname(model_save_path), exist_ok=True)
        torch.save(model.state_dict(), model_save_path)
        print(f'Model saved to {model_save_path}')


In [8]:
def test(epoch):
    file_path = '../results/cifar_10/basic_model/CNN_test.txt'
    
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    
    model.eval()
    test_loss = 0
    correct = 0
    total = 0
    
    with torch.no_grad():
        with open(file_path, 'a') as f:
            f.write(f'\nTesting after Epoch: {epoch}\n')
            
            for batch_idx, (inputs, targets) in enumerate(testloader):
                inputs, targets = inputs.to(device), targets.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, targets)

                test_loss += loss.item()
                _, predicted = outputs.max(1)
                total += targets.size(0)
                correct += predicted.eq(targets).sum().item()
                
            test_summary = f'Test Summary after Epoch {epoch}, Loss: {test_loss / len(testloader):.3f}, Accuracy: {100. * correct / total:.3f}% ({correct}/{total})\n'
            f.write(test_summary)
            
            print(test_summary)

In [9]:
Epoch = 300
for epoch in range(1, Epoch + 1):
    train(epoch)
    test(epoch)
    scheduler.step()


Epoch: 1


KeyboardInterrupt: 